# 10 — Network statistics for Task B: Hypertension (issues #16, #15)

**The task:** whether a patient will receive a first hypertension diagnosis within the next year, predicted at the end of the discharge day. Each node is one labeled prediction (3,760 discharges from 2,324 patients). Base rate of the outcome: about 14%.

This notebook describes the three patient-similarity graphs for this task:

| Graph | Built from | Role |
|---|---|---|
| **safe** | Each node's diagnoses *before* the prediction time | The honest graph we'd actually use |
| **leaky** | The patient's diagnoses over their *whole* record, including the future | A deliberate cheat, used as a control |
| **null** | The safe graph with its edges randomly rewired. Every node keeps its exact degree, and the edge weights are shuffled | A "structure means nothing" control |

It answers four questions:

1. **Shape (#11).** How many connections do nodes have, how clustered is the graph, and which nodes sit in the middle?
2. **Groups (#12).** Do Louvain and Leiden agree on the groups? Do groups differ in their new hypertension rate?
3. **Alignment with the outcome (#13).** Homophily, and Dirichlet energy of the CLMBR fingerprints.
4. **What leakage actually changes (#15).** Which links the leaky graph adds or removes, how much of each graph is a patient linked to their own other admissions, and whether any single diagnosis code gives the outcome away.

**One null graph is not enough.** Every null number is the mean over **10 differently seeded null graphs**, with the range covering the middle 95% of them.

**Data rule.** This runs locally against the on-device dataset. Only per-graph summary numbers (`results/aggregate/network_stats_task_b.csv`) and figures built from them (`results/aggregate/figures/10_*.png`) are committed. Per-node tables stay in memory.

**One code path.** All computation lives in `ehrgraph.netstats.task_analysis` and all figures in `ehrgraph.viz.network_figures`, shared by both tasks so their numbers are directly comparable (issue #17).

In [ ]:
# Standard bootstrap cell (identical in every notebook -- see notebooks/README.md):
# find the repo root, put it on the import path, and point EHRSHOT_DATA_ROOT
# at the local, on-device copy of the data.
import sys
from pathlib import Path

_repo_root = Path.cwd()
while not (_repo_root / "pyproject.toml").exists() and _repo_root != _repo_root.parent:
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root))

from scripts.local_setup import ensure_local_data  # noqa: E402

data_root = ensure_local_data()
assets = data_root / "EHRSHOT_ASSETS"
meds_reader_path = data_root / "meds_reader_omop_ehrshot"

In [ ]:
import time

import matplotlib.pyplot as plt
import pandas as pd

from ehrgraph.data.loader import open_database
from ehrgraph.eval.aggregate import write_aggregate
from ehrgraph.graphs.similarity import graph_summary
from ehrgraph.netstats.task_analysis import (
    PRIMARY_VARIANT,
    analyze_task,
    load_task_graphs,
    summary_table,
)
from ehrgraph.viz import network_figures as nf

TASK = "new_hypertension"
POSITIVE, NEGATIVE = "new hypertension", "no new hypertension"
FIG = "10"  # figure-file prefix
results_dir = _repo_root / "results" / "aggregate"
figures_dir = results_dir / "figures"
nf.apply_style()

## Build the graphs and run the analysis

`load_task_graphs` builds the safe and leaky node vectors (diagnoses only, binary per node), turns each into a mutual-10-NN similarity graph, and makes 10 degree-preserving null graphs from the safe one. It also aligns and centers the CLMBR fingerprints on the same `(patient_id, prediction_time)` key that notebook 01 validated. `analyze_task` then computes every metric in this notebook in one pass.

In [ ]:
t0 = time.time()
db = open_database(str(meds_reader_path))
tg = load_task_graphs(db, assets, TASK)
print(f"{len(tg.nodes):,} nodes, prevalence {tg.prevalence:.3f}; built in {time.time() - t0:.0f}s")

t0 = time.time()
an = analyze_task(tg, graph_variant=PRIMARY_VARIANT)
print(f"analysis: {len(an.rows)} aggregate rows in {time.time() - t0:.0f}s")
pd.DataFrame({name: graph_summary(g) for name, g in tg.graphs.items()}).T

---
# Part 1 — Shape of the graphs (issue #11)

- **Density**: the share of all possible node pairs that are connected.
- **Clustering** and **transitivity**: when two nodes share a neighbor, how often they are also connected (triadic closure, Lecture 3).
- **Degree assortativity**: whether well-connected nodes link to other well-connected nodes.
- **Path length** and **diameter**: hops between nodes in the largest connected piece. The average path is estimated from 500 sampled start nodes, which matched the exact value to 0.001 hops on Task A.

The null keeps every node's degree, so density and degree match the safe graph by construction. Its clustering and path lengths show how much structure exists beyond "who has how many links."

In [ ]:
STRUCT = [
    "density",
    "mean_degree",
    "max_degree",
    "avg_clustering",
    "transitivity",
    "degree_assortativity",
    "lcc_avg_shortest_path_hops",
    "lcc_diameter_hops_lower_bound",
    "n_components",
    "isolated_fraction",
    "largest_cc_fraction",
]
summary_table(an.rows, STRUCT)

### Degree distribution

Mutual k-NN with k = 10 caps every node at 10 links, so there are no hubs by design. The interesting part is the left side: a node with 0 or 1 links is one that nobody else counts among their closest matches.

In [ ]:
fig = nf.degree_figure(tg)
nf.save(fig, figures_dir / f"{FIG}_degree_distribution.png")
plt.show()

### Centrality

Betweenness and closeness use `distance = 1 − similarity`, and are computed exactly with igraph. Closeness and eigenvector centrality are computed inside the largest connected piece only. Eigenvector centrality *localizes* in graphs this modular: nearly all its weight lands on one dense cluster, so it is not a useful ranking of nodes.

Each AUROC below treats centrality alone as a predictor of new hypertension. 0.5 means no information. It is descriptive only: it uses every node's label.

In [ ]:
fig = nf.centrality_figure(an)
nf.save(fig, figures_dir / f"{FIG}_centrality_distributions.png")
plt.show()
summary_table(
    an.rows,
    [
        f"{m}_centrality_auroc_vs_label"
        for m in ("degree", "betweenness", "closeness", "eigenvector")
    ],
    conditions=("safe", "leaky"),
).round(3)

---
# Part 2 — Groups (issue #12)

Louvain and Leiden both maximize *modularity*: dense inside groups, sparse between them. Their agreement (adjusted Rand index, ARI) runs from 0 (chance) to 1 (identical).

The figure shows each Leiden group's new hypertension rate, for groups of at least 20 nodes. Smaller groups are noisy, and a tiny group's rate says too much about the few patients in it. **η²** is the share of outcome variation explained by group membership.

In [ ]:
summary_table(
    an.rows,
    [
        "adjusted_rand_index",
        "n_leiden_communities",
        "louvain_modularity",
        "leiden_modularity",
        "leiden_outcome_eta_squared",
        "leiden_outcome_nmi",
        "leiden_outcome_share_in_reported_communities",
    ],
)

In [ ]:
fig = nf.community_outcome_figure(tg, an, POSITIVE)
nf.save(fig, figures_dir / f"{FIG}_community_outcome_rates.png")
plt.show()

### What the graphs look like

The largest connected piece of each graph, drawn with a force-directed layout. Each dot is one node, colored only by outcome: **violet** = new hypertension, **light gray** = no new hypertension. Nothing is labeled or identifiable. Homophily shows up as violet dots clumping together; the null panel shows what no clumping looks like.

In [ ]:
fig = nf.layout_figure(tg, POSITIVE, NEGATIVE)
nf.save(
    fig, figures_dir / f"{FIG}_graph_layouts.png", dpi=80
)  # thousands of dots; keeps it < 500 KB
plt.show()

---
# Part 3 — Do the graphs line up with the outcome? (issue #13)

**Class-insensitive homophily** (Lim et al., 2021): for each outcome, how often its nodes link to same-outcome nodes, minus what random wiring would give. Random wiring scores about **0** at any prevalence, and perfect segregation scores **1**. That correction matters because Task A (Long Length of Stay) has a different prevalence (0.25).

**Dirichlet energy** asks the same question of the CLMBR fingerprints: how different a node's fingerprint is from its neighbors', normalized across graphs. **Lower means neighbors already look alike to CLMBR.**

Both use every node's label or fingerprint. They describe the graph, and must never be used to choose a model.

In [ ]:
summary_table(
    an.rows,
    [
        "edge_homophily",
        "class_insensitive_homophily",
        "same_outcome_rate_positive",
        "same_outcome_rate_negative",
        "dirichlet_energy_clmbr",
    ],
)

In [ ]:
fig = nf.homophily_energy_figure(an)
nf.save(fig, figures_dir / f"{FIG}_homophily_dirichlet.png")
plt.show()

---
# Part 4 — What does leakage actually change? (issue #15)

Three checks:

1. **Same-patient links.** A patient with several labeled admissions appears as several nodes, and those nodes tend to link to each other: their histories overlap, and in the leaky graph they are *identical*, since every admission sees the whole record. Those links share an outcome far more often than chance. So homophily is reported both with and without them. The official splits are per patient, so these links never cross train and test.
2. **Which links leakage swaps.** Links in both graphs, only in safe, and only in leaky. If leakage helps, the leaky-only links should join same-outcome nodes more often than the safe-only links they replaced.
3. **Single-code leakage.** For each diagnosis code, the share of positive vs. negative nodes whose vector contains it. In the leaky vectors a huge gap means the future record contains the answer itself.

In [ ]:
summary_table(
    an.rows,
    [
        "same_patient_edge_fraction",
        "same_patient_edge_same_outcome_rate",
        "class_insensitive_homophily",
        "class_insensitive_homophily_excl_same_patient",
        "max_single_code_outcome_gap",
    ],
)

In [ ]:
edge_sets = pd.DataFrame(
    {k: an.edge_sets[k] for k in ("shared_edges", "safe_only_edges", "leaky_only_edges")}
).T
print(f"edge Jaccard (safe vs leaky): {an.edge_sets['edge_jaccard']:.3f}")
edge_sets

In [ ]:
# The ten codes whose presence in the LEAKY vectors best separates positives
# from negatives -- code names and group rates only, no patient-level data.
an.top_leaky_codes.round(3)

In [ ]:
fig = nf.leakage_edges_figure(tg, an)
nf.save(fig, figures_dir / f"{FIG}_leakage_edges.png")
plt.show()

---
## Save the aggregate results

`write_aggregate` checks the schema and refuses patient-identifying columns before anything touches disk.

In [ ]:
out = write_aggregate(an.rows, results_dir / "network_stats_task_b.csv")
saved = pd.read_csv(out)
print(f"wrote {len(saved)} rows -> {out.relative_to(_repo_root)}")
print(sorted(p.name for p in figures_dir.glob(f"{FIG}_*.png")))
saved.groupby("condition")["metric"].count()

## Findings

Numbers from the run on Oct 2, 2026: 3,760 discharges from 2,324 patients, prevalence 0.137, primary graph configuration. "Null" means the mean of 10 rewired graphs, with the middle-95% range in brackets.

**1. The same structural picture as Task A.** Transitivity is 0.503 (safe) and 0.600 (leaky), against 0.002 for null. The average path in the largest piece is 8.4 and 9.8 hops, against 4.8 for null. The leaky graph is again more fragmented: 74% of nodes in its largest piece, against 86% for safe. Modularity is 0.93 / 0.96, against 0.56 for null. Louvain and Leiden agree (ARI 0.75 / 0.78, against 0.28 for null). Centrality again carries no outcome information (AUROC 0.45–0.56).

**2. Here, leakage visibly helps the graph line up with the outcome.**

| | safe | leaky | null |
|---|---|---|---|
| class-insensitive homophily | 0.282 | **0.332** | 0.004 [0.002–0.009] |
| same, without same-patient links | 0.136 | **0.193** | 0.004 |
| outcome variance explained by community (η²) | 0.076 | **0.123** | 0.012 [0.009–0.017] |
| Dirichlet energy (CLMBR) | 0.651 | 0.647 | 0.954 [0.950–0.956] |

Leaky homophily is 18% higher than safe, and **42% higher once same-patient links are removed**. Communities explain 60% more of the outcome. This is the opposite of Task A, where leakage added nothing.

**3. The future record contains the answer.** In the leaky vectors, ICD-9 401.9 (essential hypertension) appears in **67% of positives against 5% of negatives**, and ICD-10 I10 (the same diagnosis in the newer coding system) in 52% against 13%. The largest single-code gap is **0.62 for leaky against 0.12 for safe**. The leaky graph is literally linking patients through the diagnosis we are trying to predict.

**4. The graph dilutes even a near-answer-key.** A code present in 67% of positives moves homophily by only 0.05. Two reasons: it is 1 code among about 100 per vector, and TF-IDF *down-weights* common codes, which 401.9 is. How much leakage reaches a graph depends on how the graph is built. That is a methods point worth making in the paper, and a reason the graph-variant sweep matters.

**5. Half of Task B's homophily is the same patient, linked to themselves.** 31% of links join two discharges of the same patient, and those agree on the outcome **96%** of the time, against 76% by chance. Hypertension status rarely changes between a patient's discharges. Removing those links cuts safe homophily from 0.282 to 0.136. The official splits are per patient, so this never leaks across train and test, but it means **much of what a graph model could "borrow" in Task B is a patient's own other admission.**

**6. The links leakage adds are only slightly better, so the gain is spread thin.** Leaky-only links join same-outcome nodes 77.6% of the time; the safe-only links they replace, 76.4%, which is exactly chance (76.4%). The leaky graph's advantage comes from many small shifts across shared neighborhoods, not from one block of obviously "cheating" links.

### What this means for the project

- **Task B is where the paper's warning will show, if anywhere.** The leaky vectors carry the answer, and the graph's outcome alignment rises measurably. Project 2's prediction grid should expect the leaky-vs-safe gap to be larger here than for long stay.
- **Same-patient links need an explicit decision before Project 2.** In Task B they carry half the homophily. Report both versions, and consider a "no same-patient links" graph variant in the sweep, so a reviewer can see how much of any gain is "the same patient, again."

### Caveats

- Same as notebook 07: one graph configuration; homophily and group rates use every node's label, so they describe the graph and must never be used to choose a model.
- 9% of positives already have an I10 code *before* the prediction time in the safe vectors (2% for 401.9). This may come from how EHRSHOT defines a "new" diagnosis. It is not checked yet.

### Done when
- [x] #16: the full #11–#13 analysis repeated for Hypertension, through the same library code as Task A
- [x] #15 checks for Task B: same-patient links, leakage edge sets, single-code leakage
- [x] Only aggregate output written: `results/aggregate/network_stats_task_b.csv` and 6 figures (`10_*.png`)